About frozen lake env : https://gymnasium.farama.org/environments/toy_text/frozen_lake/

Frozen lake involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. The player may not always move in the intended direction due to the slippery nature of the frozen lake.


Action Space

The action shape is (1,) in the range {0, 3} indicating which direction to move the player.

0: Move left

1: Move down

2: Move right

3: Move up

Observation Space

The observation is a value representing the player’s current position as current_row * ncols + current_col (where both the row and col start at 0). Therefore, the observation is returned as an integer.

since we dont have a gridworld in gymnasium we use the frozen lake env with is_alippery=false to emulate a grid world. this would mean that the ahgent will transition to intended state with probability 1.

To get started with Value Iteration or Policy Iteration on a discrete environment like FrozenLake-v1, we need to know how the world works. It's the probability P(s', r | s, a)—the chance of landing in state s' with reward r after taking action a in state s.



In [13]:
import gymnasium as gym
# Create the environment (is_slippery=False makes it a deterministic grid)
env = gym.make("FrozenLake-v1", is_slippery=False)
env.reset()
print(env.unwrapped.P.keys())
print(env.unwrapped.P.values())
# The transition dynamics for state 0, action 1
# The output will look like a list of tuples: [(probability of transition, next_state, reward, terminated)].
transitions = env.unwrapped.P[0][1]
print(transitions)

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15])
dict_values([{0: [(1.0, 0, 0, False)], 1: [(1.0, 4, 0, False)], 2: [(1.0, 1, 0, False)], 3: [(1.0, 0, 0, False)]}, {0: [(1.0, 0, 0, False)], 1: [(1.0, 5, 0, True)], 2: [(1.0, 2, 0, False)], 3: [(1.0, 1, 0, False)]}, {0: [(1.0, 1, 0, False)], 1: [(1.0, 6, 0, False)], 2: [(1.0, 3, 0, False)], 3: [(1.0, 2, 0, False)]}, {0: [(1.0, 2, 0, False)], 1: [(1.0, 7, 0, True)], 2: [(1.0, 3, 0, False)], 3: [(1.0, 3, 0, False)]}, {0: [(1.0, 4, 0, False)], 1: [(1.0, 8, 0, False)], 2: [(1.0, 5, 0, True)], 3: [(1.0, 0, 0, False)]}, {0: [(1.0, 5, 0, True)], 1: [(1.0, 5, 0, True)], 2: [(1.0, 5, 0, True)], 3: [(1.0, 5, 0, True)]}, {0: [(1.0, 5, 0, True)], 1: [(1.0, 10, 0, False)], 2: [(1.0, 7, 0, True)], 3: [(1.0, 2, 0, False)]}, {0: [(1.0, 7, 0, True)], 1: [(1.0, 7, 0, True)], 2: [(1.0, 7, 0, True)], 3: [(1.0, 7, 0, True)]}, {0: [(1.0, 8, 0, False)], 1: [(1.0, 12, 0, True)], 2: [(1.0, 9, 0, False)], 3: [(1.0, 4, 0, False)]}, {0: [(1.0, 8, 0

As our first approach to solve the problem we are going to tale state value iteration.
the way value iteration works is 
1. we start with an arbitary value function, where all state values could be zero, it needs to be of the size of toatal number of possible states.

In [14]:
import numpy as np
num_states = env.observation_space.n
num_actions = env.action_space.n
state_values = np.zeros(num_states)
print(state_values)


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


We also need a discount factor to discount future rewards for mathematical simplicity. lets say we keep it 0.9 

In [15]:
gamma = 0.9

2. iteratively apply the bellman optimality equation to update the value function for each state this involves one step lookahed considering best actions to take from each state to maximize the expected future reward using the current estimate of the value function for sucessor state s, the update rule is taking maximum over all possible actions 

So there will be 2 for loops
1. for iterating over all the states
2. for iterating over all actions in that state


1. The Textbook Version (Reward Outside) In many of David Silver’s slides, you see:

V(s) = max_a [ R_s^a + gamma sum_{s' \in S} P_{ss'}^a V(s') ]

This version assumes that the reward R_s^a is "given" the moment you take action $a$ in state $s$, regardless of which $s'$ you land in.
2. The Gymnasium Version (Reward Inside)In the Gymnasium FrozenLake environment (and most real-world scenarios), the reward depends on the outcome. You only get the $+1$ if you actually land in the Goal square. If you are in the square next to the Goal, take the action "Right," but "slip" into a Hole, your reward is $0$.Therefore, the reward is part of the transition tuple $(s, a, s')$. 
   The equation looks like this:
    V(s) = \max_a \sum_{s', r} p(s', r | s, a) [r + \gamma V(s')]$$

In [16]:
for s in range(num_states):
    action_values = np.zeros(num_actions)
    for a in range(num_actions):# Inside your action loop:
        # print(env.unwrapped.P[s][a])
        for prob, next_state, reward, done in env.unwrapped.P[s][a]:
            # print(prob, next_state, reward, done)
            if done:
                future_value = 0
            else:
               future_value = state_values[next_state]
            action_values[a] += prob*(reward + gamma * future_value)
    state_values[s] = np.max(action_values)
print(state_values)
#in iteration 1 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.9 0.  0.  0.9 1.  0. ]        


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]


As we can see there has been a change  in the values of the states we neevery lowd to keep it running till the change in state values becomes 

In [17]:
theta = 0.00001
while True:
    delta = 0
    for s in range(num_states):
        v_old = state_values[s]
        action_values = np.zeros(num_actions)
        for a in range(num_actions):# Inside your action loop:
            # print(env.unwrapped.P[s][a])
            for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                # print(prob, next_state, reward, done)
                if done:
                    future_value = 0
                else:
                    future_value = state_values[next_state]
                action_values[a] += prob*(reward + gamma * future_value)
        state_values[s] = np.max(action_values)
        delta = max(delta, abs(v_old - state_values[s]))
    if(delta<theta):
        break
print(state_values.reshape(4,4))

[[0.59049 0.6561  0.729   0.6561 ]
 [0.6561  0.      0.81    0.     ]
 [0.729   0.81    0.9     0.     ]
 [0.      0.9     1.      0.     ]]


Now once you got the state values all you need to do is find the next action based on the state values table the action which leads you to the most reward giving state which is basically extracting policy out of this 

agent currently knows the "worth" of every square, but it doesn't have a Policy (a mapping from state to action). We need to create an array of 16 integers where each integer (0-3) represents the best direction to move.

The Logic: For each state, you "look ahead" at the four possible actions using your final state_values and pick the one that gives the highest value.

In [18]:
policy = np.zeros(num_states, dtype=int)

for s in range(num_states):
    action_values = np.zeros(num_actions)
    for a in range(num_actions):
        for prob, next_state, reward, done in env.unwrapped.P[s][a]:
            # Use your FINAL state_values here to see which action is best
            future_value = state_values[next_state] if not done else 0
            action_values[a] += prob * (reward + gamma * future_value)
    
    # The best action is the one with the highest Q-value
    policy[s] = np.argmax(action_values)

print("Best Actions (0:L, 1:D, 2:R, 3:U):")
print(policy.reshape(4,4))

Best Actions (0:L, 1:D, 2:R, 3:U):
[[1 2 1 0]
 [1 0 1 0]
 [2 1 1 0]
 [0 2 2 0]]


lets use this policy to run the game


In [ ]:
env_2 = env = gym.make("FrozenLake-v1", is_slippery=False, render_mode = "human")

state, _ = env.reset()
done = False
# print(f"state"{state})


while not done:
    action = policy[state] # Follow the policy you just calculated
    state, reward, terminated, truncated, _ = env.step(action)
    env.render() # If you used render_mode="human" in gym.make
    done = terminated or truncated

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2782893753.py, line 5)